# PlanetScope acquisition — Folger Deep vessel-noise study

Search, screen, order and download PlanetScope imagery for **June–August 2020–2025**,
clipped to the 99.7 km² Folger core AOI, for pairing against the Folger Deep hydrophone.

**Environment:** CryoCloud JupyterHub, ~4 GB RAM, Planet SDK v2.

---

## The two budgets

| Budget | Allocation | Cost per scene | Ceiling |
|---|---|---|---|
| **Imagery** (E&R Basic) | 3,000 km²/month | 100 km² minimum when clipping | **30 scenes/month** |
| **Scene tiles** | ~98,000 tiles/month | 64 tiles (inner box) or 196 (full AOI) at z15 | ~1,500 previews |

These are metered separately, which is what makes this workflow work. Tiles, quicklooks and
UDM2 masks are all free against the imagery budget, so **stages 1–7 cost no imagery quota at
all**. Only stage 8 spends, and it sits behind a manual gate.

The imagery budget is the scarce one: 30 scenes a month, and no amount of geometry tuning
changes it — clipping below 100 km² is charged 100 km² anyway. So every optimisation comes
from *choosing better scenes*, using the free stages to eliminate candidates first.

## Order of operations

1. Setup and AOI
2. Search — deliberately loose
3. Glint modelling from metadata
4. UDM2 cloud screening, windowed to the AOI
5. Quicklook skim
6. **Tile preview — visually confirm vessels before spending**
7. Acoustic join
8. Rank, stratify, order

## 1. Setup

In [ ]:
%pip install --quiet "planet>=2.1,<3" pandas rasterio pillow httpx

In [ ]:
import asyncio, json, math, os
from datetime import datetime, timedelta
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import httpx
import rasterio
from PIL import Image
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask, bounds as geom_bounds
from rasterio.warp import transform_geom
from planet import Auth, Session, data_filter, order_request, reporting

# ------------------------------------------------------------------ paths
AOI_PATH  = Path("folger_core_aoi.geojson")
WORK      = Path("planet_folger");            WORK.mkdir(exist_ok=True)
UDM_DIR   = WORK / "udm2_scratch";            UDM_DIR.mkdir(exist_ok=True)
PREVIEW   = WORK / "tile_previews";           PREVIEW.mkdir(exist_ok=True)

# ------------------------------------------------------------------ search
YEARS     = range(2020, 2026)
SUMMER    = (6, 9)                 # June 1 -> Sept 1, exclusive upper bound
ITEM_TYPE = "PSScene"

# Deliberately LOOSE. cloud_cover describes the whole ~637 km2 footprint; your AOI is
# ~15% of it. The free UDM2 screen in stage 4 does the real filtering. Tightening
# this is actively counterproductive - it discards scenes clear over Folger Passage.
SEARCH_CLOUD_MAX = 0.60

# ------------------------------------------------------------------ imagery quota
AOI_AREA_KM2     = 99.7
MIN_CHARGE       = 100.0           # preferred clipping minimum, per intersecting scene
MONTHLY_QUOTA    = 3000.0
CHARGE_PER_SCENE = max(AOI_AREA_KM2, MIN_CHARGE)
SCENES_PER_MONTH = int(MONTHLY_QUOTA // CHARGE_PER_SCENE)

# ------------------------------------------------------------------ tile quota
TILE_BUDGET   = 98249
TILE_ZOOM     = 15                 # 3.15 m/px here, matching PlanetScope native 3 m.
                                   # z16 costs 3.4x more tiles for pure interpolation.
PREVIEW_HALF_M = 2500              # inner box half-width: the hydrophone's short range
TILE_LEDGER   = WORK / "tile_ledger.json"

# ------------------------------------------------------------------ bundle policy
# SINGLE-RASTER ONLY. A "_udm2" bundle ships a second raster asset which may draw a
# second 100 km2 minimum - for a file the Data API gives us free in stage 4.
SINGLE_RASTER_BUNDLES = {"visual", "analytic_sr", "analytic_8b_sr",
                         "analytic", "analytic_8b"}
BUNDLE = "visual"

assert BUNDLE in SINGLE_RASTER_BUNDLES, (
    f"{BUNDLE!r} is not a known single-raster bundle. Anything ending in '_udm2' "
    f"risks a doubled charge. Allowed: {sorted(SINGLE_RASTER_BUNDLES)}")
assert not BUNDLE.endswith("_udm2"), "Multi-raster bundle rejected."
# Never set fallback_bundle: it can silently substitute a multi-raster bundle for
# older scenes, doubling the charge on exactly the acquisitions you cannot re-order.

print(f"Imagery : {SCENES_PER_MONTH} scenes/month at {CHARGE_PER_SCENE:.0f} km^2")
print(f"Tiles   : {TILE_BUDGET:,} at z{TILE_ZOOM}")
print(f"Bundle  : {BUNDLE} (single raster)")

### Authentication

Run `planet auth init` once in a CryoCloud terminal. That writes `~/.planet.json` with
restrictive permissions, so your key never enters a notebook cell or a git commit.

In [ ]:
try:
    auth = Auth.from_file()
    print("Authenticated from ~/.planet.json")
except Exception:
    key_env = os.environ.get("PL_API_KEY")
    if not key_env:
        raise SystemExit("Run `planet auth init` in a terminal, or export PL_API_KEY.")
    auth = Auth.from_key(key_env)
    print("Authenticated from PL_API_KEY")

API_KEY = auth.value        # needed for raw tile / thumbnail HTTP requests

In [ ]:
aoi = json.load(open(AOI_PATH))
if   aoi["type"] == "FeatureCollection": aoi = aoi["features"][0]["geometry"]
elif aoi["type"] == "Feature":           aoi = aoi["geometry"]
assert aoi["type"] == "Polygon"

ring = aoi["coordinates"][0]
AOI_BBOX = (min(c[0] for c in ring), min(c[1] for c in ring),
            max(c[0] for c in ring), max(c[1] for c in ring))
HYD_LON, HYD_LAT = -125.278277, 48.814200

print(f"AOI bbox: {AOI_BBOX}")

## 2. Search — deliberately loose

Six separate June–August windows combined with an OR filter. A single
`gte=2020-06-01, lte=2025-08-31` range would sweep in every winter in between.

Note `cloud_cover` is a **fraction**, not a percent. Passing `10` instead of `0.10` matches
every scene ever acquired — the commonest way to have a filter that looks like it works.

In [ ]:
search_filter = data_filter.and_filter([
    data_filter.geometry_filter(aoi),
    data_filter.range_filter("cloud_cover", lte=SEARCH_CLOUD_MAX),
    data_filter.or_filter([
        data_filter.date_range_filter("acquired",
                                      gte=datetime(y, SUMMER[0], 1),
                                      lt =datetime(y, SUMMER[1], 1))
        for y in YEARS]),
    data_filter.permission_filter(),
    data_filter.string_in_filter("quality_category", ["standard"]),
])

async def run_search():
    async with Session(auth=auth) as sess:
        res = sess.client("data").search([ITEM_TYPE],
                                         search_filter=search_filter, limit=0)
        return [i async for i in res]

items = await run_search()
json.dump(items, open(WORK / "search_results.json", "w"))
print(f"{len(items)} candidates cached (search costs no quota)")

In [ ]:
df = pd.DataFrame([{
    "id":            it["id"],
    "acquired":      pd.to_datetime(it["properties"]["acquired"]),
    "cloud_cover":   it["properties"]["cloud_cover"],
    "clear_percent": it["properties"].get("clear_percent"),
    "instrument":    it["properties"].get("instrument"),
    "sun_elevation": it["properties"].get("sun_elevation"),
    "sun_azimuth":   it["properties"].get("sun_azimuth"),
    "view_angle":    it["properties"].get("view_angle"),
    "sat_azimuth":   it["properties"].get("satellite_azimuth"),
    "thumbnail":     it["_links"].get("thumbnail"),
    "tiles_link":    it["_links"].get("tiles"),
} for it in items])

df["date"] = df["acquired"].dt.date
df["year"] = df["acquired"].dt.year
df = df.sort_values("acquired").reset_index(drop=True)

print(df.groupby("year").agg(scenes=("id","size"), days=("date","nunique")))
print(f"\ntiles link present on {df['tiles_link'].notna().sum()}/{len(df)} items")

## 3. Glint modelling — free

Over land UDM2 works well. Over water it is out of its training distribution and misreads
sun glint. But glint is deterministic geometry, predictable from metadata you already have.

The glint angle is the angle between the sensor view direction and the sun's specular
reflection direction:

$$\cos\Theta_g = \cos\theta_v\cos\theta_s - \sin\theta_v\sin\theta_s\cos(\phi_v-\phi_s)$$

Small $\Theta_g$ means the sensor is staring into the glitter pattern, which flattens wake
contrast to nothing. Near-nadir PlanetScope at 48.8°N in mid-morning summer usually lands
around 35–45°; off-nadir acquisitions toward the solar azimuth are the exception.

In [ ]:
ts   = np.radians(90.0 - df["sun_elevation"])       # solar zenith
tv   = np.radians(df["view_angle"].abs())           # sensor zenith
dphi = np.radians(df["sat_azimuth"] - df["sun_azimuth"])

df["glint_angle"] = np.degrees(np.arccos(np.clip(
    np.cos(tv)*np.cos(ts) - np.sin(tv)*np.sin(ts)*np.cos(dphi), -1, 1)))

print(df["glint_angle"].describe().round(1))

GLINT_MIN = 20          # calibrate against your first batch rather than trusting this
df = df[df["glint_angle"] > GLINT_MIN].copy()
print(f"\n{len(df)} scenes after glint screen")

## 4. UDM2 cloud screening on the AOI — free

Planet's documentation states that downloading UDM2 via the **Data API** does not count
against download quota. That is the linchpin of this whole notebook.

`cloud_cover` and `clear_percent` are computed over the entire ~637 km² footprint. Your AOI
is ~15% of that, so the scene-level number tells you very little. The failure runs both ways:
a 55%-clear scene may be spotless over Folger Passage, while a 96%-clear scene can have its
one cloud sitting directly on the hydrophone.

Reading one band in one window keeps peak memory near 10 MB regardless of scene size.

In [ ]:
UDM2_CLEAR_BAND = 1   # 1 clear, 2 snow, 3 shadow, 4 light haze,
                      # 5 heavy haze, 6 cloud, 7 confidence, 8 unusable-data mask

def aoi_clear_fraction(udm_path, aoi_geojson):
    '''Fraction of AOI pixels flagged clear. Windowed single-band read.'''
    with rasterio.open(udm_path) as src:
        geom = transform_geom("EPSG:4326", src.crs, aoi_geojson)
        win  = from_bounds(*geom_bounds(geom), src.transform)
        win  = win.round_offsets().round_lengths()
        clear = src.read(UDM2_CLEAR_BAND, window=win)
        if clear.size == 0:
            return np.nan
        inside = geometry_mask([geom], out_shape=clear.shape,
                               transform=src.window_transform(win), invert=True)
        return float((clear[inside] == 1).mean()) if inside.any() else np.nan


async def screen_udm2(item_ids, directory=None, keep_files=False):
    '''Download + score UDM2. Costs no imagery quota. Files deleted after scoring
    unless keep_files, to protect CryoCloud disk.'''
    directory = directory or UDM_DIR
    out = {}
    async with Session(auth=auth) as sess:
        cl = sess.client("data")
        for n, iid in enumerate(item_ids, 1):
            try:
                a = await cl.get_asset(ITEM_TYPE, iid, "ortho_udm2")
                await cl.activate_asset(a)
                a = await cl.wait_asset(a, max_attempts=200)
                p = await cl.download_asset(a, directory=directory,
                                            overwrite=False, progress_bar=False)
                out[iid] = aoi_clear_fraction(p, aoi)
                if not keep_files:
                    Path(p).unlink(missing_ok=True)
            except Exception as e:
                print(f"  {iid}: {type(e).__name__} {e}")
                out[iid] = np.nan
            if n % 25 == 0:
                print(f"  screened {n}/{len(item_ids)}")
    return out

df["aoi_clear"] = df["id"].map(await screen_udm2(df["id"].tolist()))

In [ ]:
corr = df[["aoi_clear", "clear_percent"]].corr().iloc[0, 1]
strict = df["aoi_clear"] >= 0.95

print(f"AOI-clear vs scene-level clear_percent correlation: {corr:.2f}")
print(f"Scenes >=95% clear over the AOI: {strict.sum()}")
print(f"  ...that a cloud_cover<=0.10 filter would have DISCARDED: "
      f"{(strict & (df['cloud_cover'] > 0.10)).sum()}   <- recovered")
print(f"Scenes passing cloud_cover<=0.10 but NOT clear over the AOI: "
      f"{((df['cloud_cover'] <= 0.10) & ~strict).sum()}   <- rejected")

df = df[strict].copy()
print(f"\n{len(df)} scenes carried forward")

## 5. Quicklook skim — free

Thumbnails run roughly 100 m/pixel across the footprint, so a boat is invisible. What they
do show is marine fog banks and broad glint sheets, both of which UDM2 misreads as clear
over water. Two minutes of eyeballing removes scenes that would fail QC later.

In [ ]:
def contact_sheet(rows, url_col="thumbnail", cols=6, thumb=200, label=True):
    imgs = []
    with httpx.Client(auth=(API_KEY, ""), timeout=30, follow_redirects=True) as c:
        for _, r in rows.iterrows():
            if not r.get(url_col):
                continue
            try:
                im = Image.open(BytesIO(c.get(r[url_col]).content)).convert("RGB")
                imgs.append(im.resize((thumb, thumb)))
            except Exception:
                pass
    if not imgs:
        return None
    nrow = -(-len(imgs) // cols)
    sheet = Image.new("RGB", (cols*thumb, nrow*thumb), "black")
    for i, im in enumerate(imgs):
        sheet.paste(im, ((i % cols)*thumb, (i // cols)*thumb))
    return sheet

contact_sheet(df.head(36))

In [ ]:
# Drop anything obviously fogged or glinted, by scene id.
FOGGED = []          # e.g. ["20230714_190112_23_2439"]
df = df[~df["id"].isin(FOGGED)].copy()
print(f"{len(df)} scenes after visual skim")

## 6. Tile preview — confirm vessels before spending

Scene tiles are metered **separately** from the imagery quota, so this stage is free against
the 30-scene budget. At z15 the ground resolution is 3.15 m/px, matching PlanetScope native.

What that resolves:

| Object | Pixels at z15 |
|---|---|
| 7 m skiff | 2.2 |
| 12 m charter | 3.8 |
| 30 m coastal freighter | 9.5 |
| Planing wake, width | 19 |
| Planing wake, length | 127 |

So this confirms **wakes and mid-size hulls**, not small boats sitting still — the same bias
the full imagery carries, arriving early enough to act on.

**Tiered, to stay inside the tile budget.** Preview the inner 5 × 5 km box first (64 tiles,
the range the hydrophone actually hears), and pull the full AOI only for scenes that survive.

Three cautions built into the code below: the tile URL comes from the item's `_links`
rather than being constructed; concurrency is capped with retry-on-429; and tiles are
composited to a single JPEG then discarded, because caching all 98k tiles would be ~4.9 GB.

In [ ]:
# If items carry no "tiles" link, paste the template from Planet's docs / the web UI here.
# Use {item_id}, {z}, {x}, {y} placeholders. Leave as None to rely on _links.
TILE_URL_TEMPLATE = None

def deg2tile(lon, lat, z):
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n)
    return x, y

def box_around(lon, lat, half_m):
    '''Small geographic box of half-width half_m metres, at this latitude.'''
    dlat = half_m / 111_206.0
    dlon = half_m / (111_320.0 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

def tile_grid(bbox, z):
    x0, y0 = deg2tile(bbox[0], bbox[3], z)     # NW corner -> min x, min y
    x1, y1 = deg2tile(bbox[2], bbox[1], z)     # SE corner -> max x, max y
    return list(range(x0, x1 + 1)), list(range(y0, y1 + 1))

INNER_BOX = box_around(HYD_LON, HYD_LAT, PREVIEW_HALF_M)
xs, ys = tile_grid(INNER_BOX, TILE_ZOOM)
print(f"Inner {2*PREVIEW_HALF_M/1000:.0f} km box: {len(xs)}x{len(ys)} = "
      f"{len(xs)*len(ys)} tiles/scene")
print(f"Affordable previews: {TILE_BUDGET // (len(xs)*len(ys)):,}")

xf, yf = tile_grid(AOI_BBOX, TILE_ZOOM)
print(f"Full AOI: {len(xf)}x{len(yf)} = {len(xf)*len(yf)} tiles/scene "
      f"-> {TILE_BUDGET // (len(xf)*len(yf)):,} previews")

In [ ]:
def _ledger():
    if TILE_LEDGER.exists():
        return json.load(open(TILE_LEDGER))
    return {"month": datetime.now().strftime("%Y-%m"), "used": 0}

def _spend(n):
    led = _ledger()
    now = datetime.now().strftime("%Y-%m")
    if led["month"] != now:                     # quota resets monthly, no rollover
        led = {"month": now, "used": 0}
    led["used"] += n
    json.dump(led, open(TILE_LEDGER, "w"))
    return led

def tile_url(row, z, x, y):
    tpl = row.get("tiles_link") or TILE_URL_TEMPLATE
    if not tpl:
        raise RuntimeError(
            "No tile URL. Items carry no '_links.tiles' and TILE_URL_TEMPLATE is unset. "
            "Copy the template from Planet's docs or the web UI - do not hand-build it.")
    return (tpl.replace("{item_id}", row["id"])
               .replace("{z}", str(z)).replace("{x}", str(x)).replace("{y}", str(y))
               .replace("{0}", "0"))


async def fetch_mosaic(row, bbox, z, concurrency=6, tile_px=256):
    '''Fetch tiles for bbox, composite to one image. Tiles are never cached.'''
    xs, ys = tile_grid(bbox, z)
    sheet  = Image.new("RGB", (len(xs)*tile_px, len(ys)*tile_px), "black")
    sem    = asyncio.Semaphore(concurrency)

    async def one(client, i, x, j, y):
        async with sem:
            for attempt in range(4):
                try:
                    r = await client.get(tile_url(row, z, x, y))
                    if r.status_code == 429:                 # rate limited
                        await asyncio.sleep(2 ** attempt)
                        continue
                    if r.status_code == 200:
                        sheet.paste(Image.open(BytesIO(r.content)).convert("RGB"),
                                    (i*tile_px, j*tile_px))
                    return
                except Exception:
                    await asyncio.sleep(2 ** attempt)

    async with httpx.AsyncClient(auth=(API_KEY, ""), timeout=60,
                                 follow_redirects=True) as client:
        await asyncio.gather(*[one(client, i, x, j, y)
                               for i, x in enumerate(xs)
                               for j, y in enumerate(ys)])
    _spend(len(xs) * len(ys))
    return sheet


async def preview_batch(rows, bbox, z, max_px=1400, quality=85):
    '''One JPEG per scene, tiles discarded. ~1-2 MB each.'''
    out = []
    for n, (_, r) in enumerate(rows.iterrows(), 1):
        try:
            im = await fetch_mosaic(r, bbox, z)
            im.thumbnail((max_px, max_px))
            p = PREVIEW / f"{r['id']}.jpg"
            im.save(p, "JPEG", quality=quality)
            out.append(p)
        except Exception as e:
            print(f"  {r['id']}: {type(e).__name__} {e}")
        if n % 20 == 0:
            print(f"  {n}/{len(rows)}   tiles used this month: {_ledger()['used']:,}")
    return out

paths = await preview_batch(df, INNER_BOX, TILE_ZOOM)
led = _ledger()
print(f"\n{len(paths)} previews written to {PREVIEW}")
print(f"Tiles used: {led['used']:,} / {TILE_BUDGET:,} "
      f"({TILE_BUDGET - led['used']:,} left this month)")

### Review the previews and record what you see

Open the JPEGs in `planet_folger/tile_previews/`. Record vessel presence in the list below.

**Before trusting a negative**, test on a scene where the hydrophone recorded an unambiguous
close passage. Tiles are 8-bit RGB rendered with a stretch tuned for land, and over dark
water that can crush wake contrast toward black. If wakes are invisible on a known-positive
scene, the rendering is the problem, not the absence of boats.

In [ ]:
VESSEL_VISIBLE = [
    # "20230714_190112_23_2439",
]

df["vessel_visible"] = df["id"].isin(VESSEL_VISIBLE)
print(f"{df['vessel_visible'].sum()} of {len(df)} scenes show a vessel or wake")

df.to_csv(WORK / "screened_candidates.csv", index=False)

## 7. Acoustic join

Two independent observations of the same instant. Crossing them gives the contingency table
that a range-detection function needs — and the **visible but not audible** cell is the one
that tells you where detection range actually falls off.

In [ ]:
det = pd.read_csv("hydrophone_detections.csv", parse_dates=["start_utc", "end_utc"])
WINDOW = timedelta(minutes=15)

df["n_detections"] = df["acquired"].apply(
    lambda t: int(((det["start_utc"] <= t + WINDOW) &
                   (det["end_utc"]   >= t - WINDOW)).sum()))
df["audible"] = df["n_detections"] > 0

print(pd.crosstab(df["vessel_visible"], df["audible"],
                  rownames=["visible"], colnames=["audible"]))

## 8. Rank, stratify, order — the only stage that spends

Ranking purely on quality concentrates scenes in whichever summer had the best weather, so
slots are allocated across years first, then ranked within each year.

Priority goes to scenes where a vessel is **visible**, since a pristine image of empty water
teaches a detector almost nothing. Five slots per thirty are held back for confirmed-quiet
controls so you can measure the false-positive rate.

In [ ]:
def score(d):
    return (d["aoi_clear"] * 100
            + d["glint_angle"].clip(upper=60) * 0.5
            - d["view_angle"].abs()
            + d["vessel_visible"] * 25            # confirmed object: dominant term
            + d["n_detections"].clip(upper=5) * 3)

df["score"] = score(df)

# Multiple satellites cross the same target on one day: near-duplicate observations.
df = df.sort_values("score", ascending=False).groupby("date", as_index=False).head(1)

targets  = df[df["vessel_visible"] | df["audible"]].copy()
controls = df[~df["vessel_visible"] & ~df["audible"]].copy()

N_CONTROL = 5
N_TARGET  = SCENES_PER_MONTH - N_CONTROL

def stratify(d, n):
    if d.empty or n <= 0:
        return d.head(0)
    share = (d["year"].value_counts(normalize=True) * n).round().astype(int)
    return pd.concat([d[d["year"] == y].nlargest(k, "score")
                      for y, k in share.items() if k > 0])

batch = pd.concat([stratify(targets, N_TARGET),
                   stratify(controls, N_CONTROL)]).sort_values("acquired")

print(f"Batch: {len(batch)} scenes = {len(batch)*CHARGE_PER_SCENE:.0f} km^2 "
      f"of {MONTHLY_QUOTA:.0f}")
print(batch.groupby("year").size())
print(f"\nwith visible vessel: {batch['vessel_visible'].sum()}   "
      f"controls: {(~batch['vessel_visible'] & ~batch['audible']).sum()}")

# --------------------------------------------------------------------- SPEND GATE
CONFIRM_ORDER = False

In [ ]:
async def place_and_download(ids, name):
    out = WORK / "downloads" / name
    out.mkdir(parents=True, exist_ok=True)
    async with Session(auth=auth) as sess:
        cl = sess.client("orders")
        req = order_request.build_request(
            name=name,
            products=[order_request.product(item_ids=ids,
                                            product_bundle=BUNDLE,
                                            item_type=ITEM_TYPE)],
            tools=[order_request.clip_tool(aoi=aoi)],   # never skip: 6x saving
        )
        with reporting.StateBar(state="creating") as bar:
            order = await cl.create_order(req)
            bar.update(state="created", order_id=order["id"])
            await cl.wait(order["id"], callback=bar.update_state, max_attempts=0)
        await cl.download_order(order["id"], directory=out,
                                overwrite=False, progress_bar=True)
    return order["id"], out


if not CONFIRM_ORDER:
    print("CONFIRM_ORDER is False - nothing ordered, no quota spent.")
else:
    ids  = batch["id"].tolist()
    name = f"folger_{datetime.now():%Y%m}"
    oid, out = await place_and_download(ids, name)
    batch.to_csv(WORK / f"manifest_{name}.csv", index=False)
    print(f"Order {oid} -> {out}")

    # Single-raster bundles ship no UDM2, so re-fetch and KEEP masks for the ordered
    # scenes. Free via the Data API. They arrive as full scenes - window at read time.
    keep = out / "udm2"; keep.mkdir(exist_ok=True)
    await screen_udm2(ids, directory=keep, keep_files=True)
    print(f"UDM2 masks retained in {keep}")

## 9. Check both budgets

In [ ]:
r = httpx.get("https://api.planet.com/auth/v1/experimental/public/my/subscriptions",
              auth=(API_KEY, ""), timeout=30)
for s in r.json():
    if s.get("state") == "active":
        print(f"{s.get('plan',{}).get('name')}: "
              f"{s.get('quota_used')} / {s.get('quota_sqkm')} km^2")

led = _ledger()
print(f"\nTiles ({led['month']}): {led['used']:,} / {TILE_BUDGET:,}")

## Next steps

`manifest_*.csv` carries the UTC `acquired` timestamp for each ordered scene — that is your
join key back to the hydrophone record. A vessel crosses the AOI in roughly 15–30 minutes,
so ±15 minutes is a reasonable starting window.

Both quotas reset on the calendar month and neither rolls over, so run stages 4–8 on a
monthly cadence rather than saving up.

**Still worth resolving:**

1. Confirm the bare `analytic_sr` / `analytic_8b_sr` bundle names with `planet orders bundles`
   before switching off `visual`. If only `_udm2` variants exist on your plan, stay on
   `visual` rather than accepting a possible doubled charge.
2. Calibrate `GLINT_MIN` against your first batch instead of trusting the 20° default.
3. Keep tiles for screening only. They are reprojected, lossy and carry no radiometry —
   anything entering a figure or a measurement should come from the ordered scene.